# BTS audit pilot on Kaggle (GPU)

This notebook runs the **Bayesian Truth Serum (BTS) audit** for multi-agent medical diagnosis on Kaggle.

- **No API key.** Models run locally via Hugging Face `transformers` (4-bit on GPU).
- **Decisive dataset:** PubMedQA (`pqa_labeled`) is the only benchmark with real abstract-section evidence (genuine H4/H5). Run it first.
- **Read the report:** `report.md` and `predictions.json` in the output directory.

**Before you run**
1. Settings → **Accelerator: GPU T4 x2** (or P100). 
2. Settings → **Internet: On** (needed to download models and/or raw data).
3. Attach this repo as a Kaggle dataset/input, **or** clone it in the setup cell below.
4. Optional but faster: attach prepared case files as a dataset so you can skip download+prepare.

> Full 300-case, 7-cell run is ~22,500 calls and can take many hours on a T4. Start with the 60-case first pass.

In [ ]:
# --- 1) Setup: locate or fetch the project ---------------------------------
import os, shutil, subprocess, sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)

# Option A: repo already present here (e.g. notebook inside the repo)
# Option B: repo attached as an input dataset under /kaggle/input/<name>/...
# Option C: git clone (requires Internet)
REPO_URL = "https://github.com/HanjalaShihab/Multi-Agent-medical-diagnosis-llm.git"

def find_pilot_root():
    candidates = [Path.cwd(), Path.cwd().parent, WORK]
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("pyproject.toml"):
            if "bts-audit" in p.read_text(errors="ignore"):
                candidates.append(p.parent)
    # common when repo is the input dataset root
    for base in list(candidates):
        if (base / "pilot" / "src" / "bts_audit").exists():
            return base
        if (base / "src" / "bts_audit").exists():
            return base.parent
    return None

root = find_pilot_root()
if root is None:
    print("Project not found locally; cloning...")
    clone_to = WORK / "Multi-Agent-medical-diagnosis-llm"
    if not clone_to.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_to)])
    root = clone_to

# Use a writable copy on Kaggle (input datasets are read-only)
if IN_KAGGLE and str(root).startswith("/kaggle/input"):
    dst = WORK / "Multi-Agent-medical-diagnosis-llm"
    if not dst.exists():
        print(f"Copying repo to {dst} ...")
        shutil.copytree(root, dst)
    root = dst

PILOT = root / "pilot"
assert (PILOT / "src" / "bts_audit").exists(), f"pilot/src/bts_audit not found under {root}"
os.chdir(PILOT)
sys.path.insert(0, str(PILOT / "src"))

# Keep caches in /kaggle/working so they survive within the session output if desired
os.environ.setdefault("HF_HOME", str(WORK / "hf_cache"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(WORK / "hf_cache"))

print("IN_KAGGLE:", IN_KAGGLE)
print("ROOT:", root)
print("PILOT:", PILOT)
print("CWD:", Path.cwd())

In [ ]:
# --- 2) Install dependencies ------------------------------------------------
# Kaggle images usually include torch/transformers; install what's missing.
# If you hit version conflicts, restart the kernel after this cell.
import sys, subprocess

def pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

# Core + parquet + HF runtime. bitsandbytes is needed for 4-bit on GPU.
pip_install(["-r", "requirements-kaggle.txt"])
print("Dependencies installed.")

In [ ]:
# --- 3) Data: download & prepare (skip if you attached prepared cases) ------
# Prepared case files go to /kaggle/working/data/cases and raw to /kaggle/working/data/raw.
# If you attached a dataset that already contains *.jsonl case files, point --data to them
# in the run cells instead of preparing here.
import subprocess, sys
from pathlib import Path

RAW = Path("/kaggle/working/data/raw")
CASES = Path("/kaggle/working/data/cases")
CASES.mkdir(parents=True, exist_ok=True)

DATASETS = ["medqa", "pubmedqa"]  # add: "mmlu", "medmcqa", "medxpertqa" as needed

# Download raw benchmarks (public, no API key)
for d in DATASETS:
    subprocess.check_call([sys.executable, "scripts/download_data.py", d, "--out", str(RAW)])

# Prepare 300-case files
prep = {
    "medqa":      ("medqa",      RAW / "medqa",      CASES / "medqa_300.jsonl"),
    "pubmedqa":   ("pubmedqa",   RAW / "pubmedqa",   CASES / "pubmedqa_300.jsonl"),
    "mmlu":       ("mmlu_med",   RAW / "mmlu",       CASES / "mmlu_med_300.jsonl"),
    "medmcqa":    ("medmcqa",    RAW / "medmcqa",    CASES / "medmcqa_300.jsonl"),
    "medxpertqa": ("medxpertqa", RAW / "medxpertqa", CASES / "medxpertqa_300.jsonl"),
}
for d in DATASETS:
    ds, raw_path, out = prep[d]
    subprocess.check_call([
        sys.executable, "-m", "bts_audit.cli", "prepare",
        "--dataset", ds, "--raw", str(raw_path), "--out", str(out), "--n", "300",
    ])

print("Cases ready in", CASES)
!ls -lh /kaggle/working/data/cases

In [ ]:
# --- 4) Environment check ---------------------------------------------------
!python -m bts_audit.cli doctor --config configs/kaggle_pubmedqa.yaml

## 5. First pass (recommended): 60 cases, 4 cells

Run this before a full sweep to check **parse rate** and runtime. Inspect `answers.jsonl` / `forecasts.jsonl` in the output folder; target >90% successful parses.

- **PubMedQA** is the decisive dataset (real evidence sections).

In [ ]:
# PubMedQA: 60-case first pass (H1,H4,H5,SYC)
!python -m bts_audit.cli run \
  --config configs/kaggle_pubmedqa.yaml \
  --data /kaggle/working/data/cases/pubmedqa_300.jsonl \
  --out  /kaggle/working/results/kaggle_pubmedqa_60 \
  --limit 60 --cells H1,H4,H5,SYC

In [ ]:
# MedQA: 60-case first pass (optional comparison)
# !python -m bts_audit.cli run \
#   --config configs/kaggle_medqa.yaml \
#   --data /kaggle/working/data/cases/medqa_300.jsonl \
#   --out  /kaggle/working/results/kaggle_medqa_60 \
#   --limit 60 --cells H1,H4,H5,SYC

## 6. Full run (300 cases, H0–H5 + SYC)

Expect ~22,500 model calls and many hours on a T4. `--resume` lets you continue after an interruption (Kaggle session timeout).

In [ ]:
# Full PubMedQA run (primary). Remove --limit/--cells to use config defaults.
# !python -m bts_audit.cli run \
#   --config configs/kaggle_pubmedqa.yaml \
#   --data /kaggle/working/data/cases/pubmedqa_300.jsonl \
#   --out  /kaggle/working/results/kaggle_pubmedqa_300 \
#   --resume

In [ ]:
# Full MedQA run
# !python -m bts_audit.cli run \
#   --config configs/kaggle_medqa.yaml \
#   --data /kaggle/working/data/cases/medqa_300.jsonl \
#   --out  /kaggle/working/results/kaggle_medqa_300 \
#   --resume

In [ ]:
# --- 7) Re-analyze or inspect results ---------------------------------------
# from pathlib import Path
# run = Path('/kaggle/working/results/kaggle_pubmedqa_60')
# print((run/'report.md').read_text()[:4000])
# !python -m bts_audit.cli analyze --run /kaggle/working/results/kaggle_pubmedqa_60

In [ ]:
# --- 8) Package results for download ----------------------------------------
# Kaggle saves /kaggle/working as notebook output.
!cd /kaggle/working && zip -rq bts_results.zip results 2>/dev/null || true
!ls -lh /kaggle/working/bts_results.zip 2>/dev/null || echo "No results directory yet"

## Notes

- **OOM / slow:** keep `max_loaded_models: 1`, `load_in_4bit: true`, lower `max_new_tokens`, or use a smaller model set in a copy of the config. A 16 GB T4 should fit one 7B 4-bit model at a time.
- **Resume:** always pass `--resume` for long runs; answers/forecasts are written incrementally.
- **Parse rate:** if low, shorten prompts or switch to more instruction-tuned models.
- **Interpretation:** negative ΔAUC in H1–H3 is expected (popularity effect), not a bug. See `pilot/docs/PILOT_GUIDE.md`.
- **CPU later:** use `configs/cpu_medqa.yaml` / `configs/cpu_pubmedqa.yaml` with `requirements-cpu.txt` (no 4-bit, small models, small limits).